# 03 — Pandas Advanced

**Topics:** Window functions, MultiIndex, reshape (pivot/melt/stack/unstack), time series.

**Reference:** [pandas documentation](https://pandas.pydata.org/docs/)

**Dataset:** Retail e-commerce transactions — order-level data with dates, products, customers.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

# Online Retail dataset: ~500k transactions from a UK retailer (2010-2011)
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame

# Light preprocessing — do not modify
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['CustomerID'] = retail['CustomerID'].astype(int)
print(retail.shape)
retail.head()

---
## Exercise 1 — Time Series Resampling

**Business question:** Build a weekly revenue time series to identify seasonality.

1. Resample `retail` to **weekly frequency** (`'W'`), summing `Revenue` and counting unique `InvoiceNo` per week.
2. Result: DataFrame `weekly_sales` with columns `Revenue` and `InvoiceCount`, indexed by week-end date.
3. Add a column `revenue_wow_pct`: week-over-week % change in Revenue (round to 2dp). First row will be NaN — leave it.
4. Add `revenue_4w_rolling_mean`: 4-week rolling mean of Revenue.

In [ ]:
def build_weekly_sales(df: pd.DataFrame) -> pd.DataFrame:
    """
    Weekly resampled revenue + invoice count.
    Adds revenue_wow_pct and revenue_4w_rolling_mean columns.
    """
    # YOUR CODE HERE
    pass

weekly_sales = build_weekly_sales(retail)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(weekly_sales.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
assert set(['Revenue', 'InvoiceCount', 'revenue_wow_pct', 'revenue_4w_rolling_mean']).issubset(weekly_sales.columns)
assert weekly_sales['Revenue'].sum() > 0
assert weekly_sales['revenue_4w_rolling_mean'].notna().sum() >= len(weekly_sales) - 3, "Rolling mean should have at most 3 NaNs"
wow = weekly_sales['Revenue'].pct_change().round(2)
assert weekly_sales['revenue_wow_pct'].dropna().equals(wow.dropna()), "WoW % change calculation is off"
print(f"✓ Exercise 1 passed — {len(weekly_sales)} weeks")

---
## Exercise 2 — Window Functions (Rolling & Expanding)

**Business question:** Track each customer's revenue trajectory to detect engagement trends.

1. Create `customer_monthly`: group by `CustomerID` and month (period), summing `Revenue`. Result should be a DataFrame with columns `CustomerID`, `Month`, `Revenue`.
2. For each customer, add:
   - `rolling_3m_avg`: 3-month rolling average of their monthly revenue (min_periods=1).
   - `cumulative_revenue`: expanding (cumulative) sum of their revenue.
   - `revenue_rank_in_month`: rank of this customer's revenue within each month (dense rank, ascending).
3. Result: `customer_monthly` DataFrame with all 5 columns + the 3 derived ones.

In [ ]:
def build_customer_monthly(df: pd.DataFrame) -> pd.DataFrame:
    """
    Monthly revenue per customer with rolling avg, cumulative sum, monthly rank.
    """
    # YOUR CODE HERE
    pass

customer_monthly = build_customer_monthly(retail)

In [ ]:
# --- ASSERTIONS ---
for col in ['rolling_3m_avg', 'cumulative_revenue', 'revenue_rank_in_month']:
    assert col in customer_monthly.columns, f"Missing {col}"

# cumulative_revenue must be monotonically increasing per customer
sample_cust = customer_monthly['CustomerID'].iloc[0]
cust_rows = customer_monthly[customer_monthly['CustomerID'] == sample_cust].sort_values('Month')
assert cust_rows['cumulative_revenue'].is_monotonic_increasing, "Cumulative revenue must increase"

# rank check: min rank in any month is 1
assert customer_monthly['revenue_rank_in_month'].min() == 1
print(f"✓ Exercise 2 passed — {len(customer_monthly)} customer-month rows")

---
## Exercise 3 — MultiIndex

**Business question:** Build a hierarchically indexed report for the merchandising team.

1. Create a MultiIndex DataFrame `product_country_stats` with index levels `(Country, StockCode)` and columns: `total_revenue`, `total_quantity`, `avg_unit_price`, `transaction_count`.
2. Sort index lexicographically.
3. Using `.xs()`, extract all rows for `Country == 'United Kingdom'` into `uk_products`.
4. Using `.loc[]` with MultiIndex slicing, extract rows where Country is between 'France' and 'Germany' (alphabetical) into `fg_products`.

In [ ]:
def build_product_country_stats(df: pd.DataFrame):
    """
    Returns:
      product_country_stats: MultiIndex DataFrame (Country, StockCode)
      uk_products: cross-section for UK
      fg_products: slice France to Germany
    """
    # YOUR CODE HERE
    pass

product_country_stats, uk_products, fg_products = build_product_country_stats(retail)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(product_country_stats.index, pd.MultiIndex)
assert product_country_stats.index.names == ['Country', 'StockCode']
assert list(product_country_stats.columns) == ['total_revenue', 'total_quantity', 'avg_unit_price', 'transaction_count']
assert product_country_stats.index.is_monotonic_increasing, "Index must be sorted"

assert 'Country' not in uk_products.index.names, "xs() should drop the cross-section level"
assert len(uk_products) == product_country_stats.loc['United Kingdom'].shape[0]

fg_countries = fg_products.index.get_level_values('Country').unique()
assert all('France' <= c <= 'Germany' for c in fg_countries)
print(f"✓ Exercise 3 passed — {len(product_country_stats)} (country, product) pairs")

---
## Exercise 4 — Melt & Stack/Unstack

**Business question:** Transform the data between wide and long formats for different reporting tools.

First, run the setup cell to create a wide-format summary.

In [ ]:
# Setup — do not modify
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
top5_countries = retail.groupby('Country')['Revenue'].sum().nlargest(5).index
wide_summary = (
    retail[retail['Country'].isin(top5_countries)]
    .groupby(['Country', 'Month'])['Revenue']
    .sum()
    .unstack('Month')
    .fillna(0)
    .round(2)
)
print(wide_summary.shape)
wide_summary.head()

1. Melt `wide_summary` into a long-format DataFrame `long_summary` with columns: `Country`, `Month`, `Revenue`. Reset index before melting.
2. From `long_summary`, use `stack/unstack` to recreate a wide DataFrame `restacked` identical in structure to `wide_summary`.
3. Verify `restacked` equals `wide_summary` (within floating point tolerance).

In [ ]:
def reshape_revenue(wide: pd.DataFrame):
    """
    Returns (long_summary, restacked).
    long_summary: columns [Country, Month, Revenue]
    restacked: same structure as wide
    """
    # YOUR CODE HERE
    pass

long_summary, restacked = reshape_revenue(wide_summary)

In [ ]:
# --- ASSERTIONS ---
assert list(long_summary.columns) == ['Country', 'Month', 'Revenue']
assert len(long_summary) == wide_summary.shape[0] * wide_summary.shape[1]
assert restacked.shape == wide_summary.shape
assert np.allclose(restacked.values, wide_summary.values, atol=0.01), "Restacked must match original"
print(f"✓ Exercise 4 passed — long format: {len(long_summary)} rows")

---
## Exercise 5 — Time Series: Date Features & Lag Variables

**Business question:** Prepare time series features for a forecasting model.

Using `weekly_sales` from Exercise 1:

1. Add calendar features: `week_of_year`, `month`, `quarter`, `is_december` (bool).
2. Add lag features: `lag_1` (1-week lag of Revenue), `lag_4` (4-week lag), `lag_52` (52-week lag).
3. Add `revenue_momentum`: rolling 4-week mean minus rolling 8-week mean (trend signal).
4. Return a copy of `weekly_sales` with all added features. Do not drop rows with NaN lags.

In [ ]:
def add_ts_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add calendar features, lag variables, and momentum to weekly_sales.
    """
    # YOUR CODE HERE
    pass

weekly_features = add_ts_features(weekly_sales)

In [ ]:
# --- ASSERTIONS ---
for col in ['week_of_year', 'month', 'quarter', 'is_december', 'lag_1', 'lag_4', 'lag_52', 'revenue_momentum']:
    assert col in weekly_features.columns, f"Missing column: {col}"

assert weekly_features['is_december'].dtype == bool
assert weekly_features['month'].between(1, 12).all()
assert weekly_features['quarter'].between(1, 4).all()

# Lag check: lag_1 at position 1 must equal Revenue at position 0
assert weekly_features['lag_1'].iloc[1] == weekly_features['Revenue'].iloc[0]
assert weekly_features['lag_4'].iloc[4] == weekly_features['Revenue'].iloc[0]

print(f"✓ Exercise 5 passed")

---
## Exercise 6 — RFM Segmentation with MultiIndex

**Business question:** Build an RFM (Recency, Frequency, Monetary) customer segmentation table — a standard CRM analytics task.

Reference date: last date in the dataset + 1 day.

1. Compute for each `CustomerID`:
   - `Recency`: days since their last purchase.
   - `Frequency`: number of unique invoices.
   - `Monetary`: total revenue.
2. Score each metric 1–4 using `pd.qcut` (quartiles). R score: lower recency = higher score (reversed). F and M: higher = higher score.
3. Create `RFM_Segment` by concatenating the three scores as strings (e.g., `'444'` = best customer).
4. Classify each customer:
   - `'Champions'`: RFM_Segment starts with '4' and total score ≥ 10
   - `'At Risk'`: R_Score == 1
   - `'Loyalists'`: F_Score >= 3 and M_Score >= 3
   - `'Others'`: everything else
5. Assign to `rfm`. Output must have `CustomerID` as index.

In [ ]:
def build_rfm(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build RFM table with scores and segment labels.
    CustomerID as index.
    """
    # YOUR CODE HERE
    pass

rfm = build_rfm(retail)

In [ ]:
# --- ASSERTIONS ---
assert rfm.index.name == 'CustomerID'
for col in ['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Segment', 'Segment']:
    assert col in rfm.columns, f"Missing: {col}"

assert rfm['R_Score'].between(1, 4).all(), "Scores must be 1-4"
assert rfm['RFM_Segment'].str.len().eq(3).all(), "RFM_Segment must be 3 chars"
assert set(rfm['Segment'].unique()).issubset({'Champions', 'At Risk', 'Loyalists', 'Others'})

# Champions must have R_Score == 4
assert rfm[rfm['Segment'] == 'Champions']['R_Score'].eq(4).all(), "Champions must have R_Score == 4 (from '4xx' segment)"

print(f"✓ Exercise 6 passed")
print(rfm['Segment'].value_counts())

---
## Exercise 7 — Custom Window: Session-Based Aggregation

**Business question:** Define customer 'sessions' (purchases within 30 days of each other) and compute session-level metrics. This is an advanced window operation that does NOT have a built-in pandas method.

1. For each customer, sort transactions by `InvoiceDate`.
2. Define a new session when the gap to the previous transaction exceeds 30 days.
3. Assign a `session_id` (integer, starting from 1 per customer).
4. Aggregate to `session_stats`: one row per `(CustomerID, session_id)` with:
   - `session_start`: first invoice date in session
   - `session_end`: last invoice date
   - `session_revenue`: total revenue
   - `n_invoices`: unique invoice count
   - `session_duration_days`: days from start to end
5. Assign to `session_stats`. Index: `(CustomerID, session_id)` MultiIndex.

In [ ]:
def build_session_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Session-based aggregation with 30-day gap threshold.
    Returns session_stats with MultiIndex (CustomerID, session_id).
    """
    # YOUR CODE HERE
    pass

session_stats = build_session_stats(retail)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(session_stats.index, pd.MultiIndex)
assert session_stats.index.names == ['CustomerID', 'session_id']
for col in ['session_start', 'session_end', 'session_revenue', 'n_invoices', 'session_duration_days']:
    assert col in session_stats.columns, f"Missing: {col}"

assert (session_stats['session_duration_days'] >= 0).all()
assert (session_stats['n_invoices'] >= 1).all()
assert session_stats['session_revenue'].sum() > 0

# Session IDs must start from 1 per customer
min_session = session_stats.groupby('CustomerID').apply(lambda x: x.index.get_level_values('session_id').min())
assert (min_session == 1).all(), "Session IDs must start from 1 per customer"

print(f"✓ Exercise 7 passed — {len(session_stats)} sessions across {session_stats.index.get_level_values('CustomerID').nunique()} customers")